In [1]:
import os 
from dotenv import load_dotenv

# Langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

C:\Users\Venkata\AppData\Local\Temp\ipykernel_6088\2384445923.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_KEY")
jina_key = os.getenv("JINA_KEY")

In [4]:
DATA_FILE_PATH = os.path.join("data", "hr_policies.txt")

### Data Ingestion

In [5]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2598

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


### Langchain document

In [6]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [7]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


#### Splitting the data

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [9]:
len(chunks)

9

In [10]:
print("\n--- Splitting document into chunks ---")
print(chunks[0].page_content)


--- Splitting document into chunks ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)


### Store data in vector DB

In [11]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


In [13]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, embeddings_model)
print("Chunks added to vectorstore", vector_store.index.ntotal)

Chunks added to vectorstore 9


In [14]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### Tools

In [15]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### Data Retrival

In [16]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [17]:
tr = llm.invoke("Hey what is the leave policy")

In [18]:
tr.content

'Sure!\u202fLeave policies can vary a lot from one organization (or country) to another, but most companies include a few common types of leave. Below is a quick overview of the typical categories you’ll see, along with some general guidelines. If you have a specific company or jurisdiction in mind, let me know and I can tailor the details further.\n\n---\n\n## 1. **Annual / Vacation Leave**\n| Feature | Typical Details |\n|---------|-----------------|\n| **Accrual** | Usually accrues based on length of service (e.g., 1–2\u202fdays per month) or is granted as a lump‑sum at the start of the year. |\n| **Maximum Balance** | Many firms cap the amount you can carry over (e.g., 5–10\u202fdays) or require you to use it within the year. |\n| **Request Process** | Submit a request through HR or an internal portal; approval often depends on workload and manager discretion. |\n| **Paid/Unpaid** | Generally paid. Some companies offer “use‑it‑or‑lose‑it” policies. |\n\n---\n\n## 2. **Sick Leave**\

In [19]:
from langchain.agents import create_agent  

In [20]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [21]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [22]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)


In [23]:
response

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='40beb351-2b4f-4346-9af6-54e1a290bec1'),
  AIMessage(content='I’m the friendly HR assistant here at **Acme\u202fCrop**. How can I help you today?', additional_kwargs={'reasoning_content': 'User asks: "tell me which org you work for". The assistant is a friendly HR assistant for Acme Crop. Should answer that I work for Acme Crop. No need to search policy. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 208, 'total_tokens': 283, 'completion_time': 0.160650303, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.008808344, 'prompt_tokens_details': None, 'queue_time': 0.070719552, 'total_time': 0.169458647}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_df9620fe21', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00153-2fbe-7b91-b

In [24]:
response["messages"][-1].content

'I’m the friendly HR assistant here at **Acme\u202fCrop**. How can I help you today?'

In [25]:
print(response["messages"][-1].content)

I’m the friendly HR assistant here at **Acme Crop**. How can I help you today?
